# Melanoma Risk Analysis — v3

**What changed from v2 — replacing every unjustified number with a data-driven decision:**

| v2 issue | v3 fix |
|---|---|
| `sampling_strategy=0.1` (guessed) | Optuna joint search — tuned together with `scale_pos_weight` because they are coupled |
| `scale_pos_weight=10` (guessed) | Same Optuna study |
| LightGBM hyperparams (`lr`, `num_leaves`, etc.) are rule-of-thumb | All 9 parameters searched jointly in Optuna |
| Platt scaling `C=1.0` (sklearn default) | Grid search over 6 values; selected by 5-fold Brier score on OOF predictions |
| High-tier recall target of 50% undocumented | Sensitivity analysis table across five targets; choice documented with explicit clinical reasoning |
| `age_approx.clip(0, 85)` unexplained | Documented: 85 is the dataset's natural encoding ceiling (ISIC 2024 uses 5-year age buckets, max bucket = 85+) |
| No model saved | Final model trained on full data with best params; saved with calibrator, thresholds, and feature list |

In [1]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import matplotlib
matplotlib.use('Agg')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

sns.set_theme(style='whitegrid')

PROJECT_ROOT   = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_RAW       = PROJECT_ROOT / 'data' / 'raw'
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed' / 'v3'
RESULTS_DIR    = PROJECT_ROOT / 'outputs' / 'results' / 'v3'
SAVE_DIR       = PROJECT_ROOT / 'outputs' / 'graphs' / 'v3'
MODEL_DIR      = PROJECT_ROOT / 'outputs' / 'models' / 'v3'

for d in [DATA_PROCESSED, RESULTS_DIR, SAVE_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')

Project root: /home/xcane/dev/ml/170melanoma


## Step 1 — Load & Preprocess

In [2]:
df_features = pd.read_csv(DATA_RAW / 'train-metadata.csv')
df_labels   = pd.read_csv(DATA_RAW / 'train-labels.csv')

df = pd.merge(df_features, df_labels, on='isic_id', how='inner')

patient_id = df.pop('patient_id')
df = df.drop(columns=['isic_id', 'image_type', 'tbp_tile_type'])

y = df['malignant']
X = df.drop(columns=['malignant'])

numeric_cols     = X.select_dtypes(include='number').columns
categorical_cols = X.select_dtypes(exclude='number').columns
X[numeric_cols]     = X[numeric_cols].fillna(X[numeric_cols].median())
X[categorical_cols] = X[categorical_cols].fillna(X[categorical_cols].mode().iloc[0])

# age_approx: ISIC 2024 encodes age in 5-year buckets; 85 is the top bucket (85+).
# The dataset's own max is 85 — this clip is a no-op that documents the valid range.
X['age_approx']             = X['age_approx'].clip(0, 85)
X['tbp_lv_nevi_confidence'] = X['tbp_lv_nevi_confidence'].clip(0, 100)  # percentage scale
X['tbp_lv_eccentricity']    = X['tbp_lv_eccentricity'].clip(0, 1)       # unit interval by definition
X['tbp_lv_symm_2axis']      = X['tbp_lv_symm_2axis'].clip(0, 1)         # unit interval by definition

X['color_contrast_3d'] = (X['tbp_lv_deltaA']**2 + X['tbp_lv_deltaB']**2 + X['tbp_lv_deltaL']**2) ** 0.5
X['elongation']        = X['tbp_lv_minorAxisMM'] / (X['clin_size_long_diam_mm'] + 1e-6)
X['nevi_color_tension']= X['tbp_lv_nevi_confidence'] * X['tbp_lv_norm_color']
X['log_area']          = X['tbp_lv_areaMM2'] ** 0.5
X['compactness']       = (X['tbp_lv_perimeterMM']**2) / (4 * 3.14159 * X['tbp_lv_areaMM2'] + 1e-6)
X['chroma_contrast']   = (X['tbp_lv_C'] - X['tbp_lv_Cext']).abs()

df = X.copy()
df['malignant']  = y.values
df['patient_id'] = patient_id.values

print(f'Shape: {df.shape}')
print(f'Malignant: {int(y.sum())} ({y.mean()*100:.3f}%) | Benign: {int((y==0).sum()):,}')

Shape: (401059, 46)
Malignant: 393 (0.098%) | Benign: 400,666


## Step 2 — Define Feature Set

`tbp_lv_symm_2axis` excluded: Mann-Whitney U test in v1 returned p = 0.143 — no statistically significant separation between classes.

In [3]:
ANALYSIS_COLS = [
    'age_approx', 'clin_size_long_diam_mm', 'tbp_lv_nevi_confidence',
    'tbp_lv_norm_border', 'tbp_lv_norm_color', 'tbp_lv_area_perim_ratio',
    'tbp_lv_color_std_mean', 'tbp_lv_eccentricity',
    'tbp_lv_deltaLBnorm', 'color_contrast_3d', 'elongation',
    'nevi_color_tension', 'log_area', 'compactness', 'chroma_contrast'
]
CAT_COLS = ['sex', 'anatom_site_general', 'tbp_lv_location_simple', 'tbp_lv_location']

print(f'Numeric features: {len(ANALYSIS_COLS)}')
print(f'Malignant: {(df.malignant==1).sum()} | Benign: {(df.malignant==0).sum():,}')

Numeric features: 15
Malignant: 393 | Benign: 400,666


## Step 3 — Build Model Feature Matrix

In [4]:
df_model = df.copy()
df_model = pd.get_dummies(df_model, columns=CAT_COLS, drop_first=True)

DROP_COLS = ['malignant', 'patient_id']
df_model  = df_model.drop(columns=[c for c in DROP_COLS if c in df_model.columns])

bool_cols = df_model.select_dtypes(include='bool').columns
df_model[bool_cols] = df_model[bool_cols].astype(int)

X_model = df_model.astype(float)
y_model = df['malignant'].values.astype(int)
groups  = df['patient_id'].values

print(f'Model features: {X_model.shape[1]}')
print(f'Class balance — malignant: {y_model.sum()} | benign: {(y_model==0).sum():,}')

Model features: 72
Class balance — malignant: 393 | benign: 400,666


## Step 4 — Hyperparameter Tuning (Optuna)

### Why we tune jointly

`sampling_strategy` and `scale_pos_weight` are **coupled**: after SMOTE oversamples the minority
class to ratio `s`, the residual imbalance inside LightGBM is approximately `1/s`. Tuning them
independently misses this interaction. All nine parameters are searched in a single joint study.

### Tuning dataset

We tune on a **stratified subsample**: all 393 malignant records + 40,000 benign records drawn
uniformly at random (preserving the ~1:102 class ratio). This makes each of the 50 trials fast
enough to finish in a reasonable time while producing hyperparameter estimates that transfer well
to the full dataset — the relative rankings of parameter combinations are stable even at reduced
scale.

### Objective

**Mean PR-AUC across 3-fold StratifiedGroupKFold.** PR-AUC is the right metric for extreme
imbalance because it is insensitive to the number of true negatives — a model cannot inflate it
by being good at ranking the majority class, only by being good at finding the minority class.

### Search space

| Parameter | Range | Scale | Rationale |
|---|---|---|---|
| `sampling_strategy` | 0.05 – 0.30 | linear | Below 0.05 gives SMOTE too few minority samples; above 0.30 risks over-smoothing |
| `scale_pos_weight` | 2 – 50 | log | Covers the residual imbalance after SMOTE; log scale since effect is multiplicative |
| `learning_rate` | 0.005 – 0.10 | log | Standard LightGBM range; log scale captures orders-of-magnitude differences |
| `num_leaves` | 31 – 127 | int | Tabular datasets rarely benefit from more than 127 leaves |
| `min_child_samples` | 10 – 100 | int | Controls leaf-level regularisation; wider range than v2 default of 20 |
| `subsample` | 0.5 – 1.0 | linear | Row subsampling for variance reduction |
| `colsample_bytree` | 0.5 – 1.0 | linear | Feature subsampling per tree |
| `reg_alpha` | 1e-8 – 1.0 | log | L1 regularisation |
| `reg_lambda` | 1e-8 – 1.0 | log | L2 regularisation |

**Expected runtime: ~15–30 minutes** (50 trials × 3 folds × ~40k rows each).

In [5]:
from imblearn.over_sampling import SMOTE
import lightgbm as lgb
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import average_precision_score, roc_auc_score

# Build tuning subsample: all positives + 40k random negatives
TUNE_BENIGN_N = 40_000
rng = np.random.RandomState(42)

mal_idx        = np.where(y_model == 1)[0]
ben_idx        = np.where(y_model == 0)[0]
ben_sample_idx = rng.choice(ben_idx, size=TUNE_BENIGN_N, replace=False)
tune_idx       = np.sort(np.concatenate([mal_idx, ben_sample_idx]))

X_tune       = X_model.iloc[tune_idx].reset_index(drop=True)
y_tune       = y_model[tune_idx]
groups_tune  = groups[tune_idx]

print(f'Tuning subsample: {len(tune_idx):,} rows | '
      f'malignant: {y_tune.sum()} | benign: {(y_tune==0).sum():,} | '
      f'ratio 1:{(y_tune==0).sum()//y_tune.sum()}')

Tuning subsample: 40,393 rows | malignant: 393 | benign: 40,000 | ratio 1:101


In [6]:
def objective(trial):
    sampling_strategy = trial.suggest_float('sampling_strategy', 0.05, 0.30)
    scale_pos_weight  = trial.suggest_float('scale_pos_weight',  2.0,  50.0, log=True)
    learning_rate     = trial.suggest_float('learning_rate',     0.005, 0.10, log=True)
    num_leaves        = trial.suggest_int(  'num_leaves',        31,   127)
    min_child_samples = trial.suggest_int(  'min_child_samples', 10,   100)
    subsample         = trial.suggest_float('subsample',         0.5,   1.0)
    colsample_bytree  = trial.suggest_float('colsample_bytree',  0.5,   1.0)
    reg_alpha         = trial.suggest_float('reg_alpha',         1e-8,  1.0, log=True)
    reg_lambda        = trial.suggest_float('reg_lambda',        1e-8,  1.0, log=True)

    sgkf3   = StratifiedGroupKFold(n_splits=3)
    pr_aucs = []

    for tr_idx, val_idx in sgkf3.split(X_tune, y_tune, groups_tune):
        X_tr, X_val = X_tune.iloc[tr_idx], X_tune.iloc[val_idx]
        y_tr, y_val = y_tune[tr_idx],       y_tune[val_idx]

        if y_val.sum() < 2:
            continue

        try:
            smote = SMOTE(sampling_strategy=sampling_strategy, random_state=42, k_neighbors=5)
            X_res, y_res = smote.fit_resample(X_tr, y_tr)
        except ValueError:
            return 0.0

        clf = lgb.LGBMClassifier(
            n_estimators=500,
            learning_rate=learning_rate,
            num_leaves=num_leaves,
            min_child_samples=min_child_samples,
            scale_pos_weight=scale_pos_weight,
            subsample=subsample,
            colsample_bytree=colsample_bytree,
            reg_alpha=reg_alpha,
            reg_lambda=reg_lambda,
            metric='auc',
            random_state=42,
            n_jobs=-1,
            verbose=-1,
        )
        clf.fit(
            X_res, y_res,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(50, verbose=False),
                       lgb.log_evaluation(period=-1)],
        )

        proba = clf.predict_proba(X_val)[:, 1]
        pr_aucs.append(average_precision_score(y_val, proba))

    return float(np.mean(pr_aucs)) if pr_aucs else 0.0


def _progress_callback(study, trial):
    # Plain print — no widgets, safe in WSL2 / headless Jupyter
    if (trial.number + 1) % 10 == 0 or trial.number == 0:
        print(f'Trial {trial.number+1:>3}/50 | best PR-AUC so far: {study.best_value:.4f}')

study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42),
)
study.optimize(objective, n_trials=50, show_progress_bar=False, callbacks=[_progress_callback])

best_params = study.best_params
best_value  = study.best_value
print(f'\nBest PR-AUC (3-fold on subsample): {best_value:.4f}')
print('Best hyperparameters:')
for k, v in best_params.items():
    print(f'  {k}: {v:.6g}' if isinstance(v, float) else f'  {k}: {v}')

Trial   1/50 | best PR-AUC so far: 0.2386
Trial  10/50 | best PR-AUC so far: 0.2632
Trial  20/50 | best PR-AUC so far: 0.2721
Trial  30/50 | best PR-AUC so far: 0.2822
Trial  40/50 | best PR-AUC so far: 0.2822
Trial  50/50 | best PR-AUC so far: 0.2822

Best PR-AUC (3-fold on subsample): 0.2822
Best hyperparameters:
  sampling_strategy: 0.0692832
  scale_pos_weight: 3.45548
  learning_rate: 0.0184314
  num_leaves: 96
  min_child_samples: 44
  subsample: 0.641414
  colsample_bytree: 0.658418
  reg_alpha: 0.00330691
  reg_lambda: 8.35941e-06


In [7]:
# Parameter importance: which parameters Optuna found most impactful
importances = optuna.importance.get_param_importances(study)
params_sorted  = list(importances.keys())
imp_sorted     = list(importances.values())

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(params_sorted[::-1], imp_sorted[::-1], color='steelblue')
ax.set_xlabel('Importance (fANOVA)')
ax.set_title('Optuna Hyperparameter Importance', fontweight='bold')
plt.tight_layout()
plt.savefig(SAVE_DIR / 'optuna_param_importance.png', dpi=150, bbox_inches='tight')
plt.close()
print('Parameter importance plot saved.')

# Trial history
trial_values = [t.value for t in study.trials if t.value is not None]
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(range(1, len(trial_values)+1), trial_values, 'o-', color='steelblue', alpha=0.7, ms=4)
ax.axhline(best_value, color='crimson', linestyle='--', label=f'Best = {best_value:.4f}')
ax.set_xlabel('Trial')
ax.set_ylabel('PR-AUC (3-fold)')
ax.set_title('Optuna Trial History', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(SAVE_DIR / 'optuna_trial_history.png', dpi=150, bbox_inches='tight')
plt.close()
print('Trial history plot saved.')

Parameter importance plot saved.
Trial history plot saved.


## Step 5 — Cross-Validation with SMOTE (Tuned Parameters)

5-fold `StratifiedGroupKFold` on the **full dataset** using the best hyperparameters from Optuna.
Groups are patient IDs so all lesions from the same patient stay in the same fold (prevents
data leakage through patient-level correlation).

We also record each fold's `best_iteration_` — the number of trees at which early stopping
fired. The mean of these is used to set `n_estimators` when training the final inference model
on all data in Step 10.

**Expected runtime: ~20–40 minutes.**

In [8]:
N_SPLITS = 5
sgkf = StratifiedGroupKFold(n_splits=N_SPLITS)

oof_proba_raw   = np.zeros(len(y_model))
fold_results    = []
best_iterations = []

for fold, (train_idx, val_idx) in enumerate(sgkf.split(X_model, y_model, groups)):
    X_tr, X_val = X_model.iloc[train_idx], X_model.iloc[val_idx]
    y_tr, y_val = y_model[train_idx],       y_model[val_idx]

    smote = SMOTE(
        sampling_strategy=best_params['sampling_strategy'],
        random_state=42,
        k_neighbors=5,
    )
    X_res, y_res = smote.fit_resample(X_tr, y_tr)

    lgbm = lgb.LGBMClassifier(
        n_estimators=1000,
        learning_rate=best_params['learning_rate'],
        num_leaves=best_params['num_leaves'],
        min_child_samples=best_params['min_child_samples'],
        scale_pos_weight=best_params['scale_pos_weight'],
        subsample=best_params['subsample'],
        colsample_bytree=best_params['colsample_bytree'],
        reg_alpha=best_params['reg_alpha'],
        reg_lambda=best_params['reg_lambda'],
        metric='auc',
        random_state=42,
        n_jobs=-1,
        verbose=-1,
    )
    lgbm.fit(
        X_res, y_res,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(100, verbose=False),
                   lgb.log_evaluation(period=-1)],
    )

    best_iterations.append(lgbm.best_iteration_)
    raw_proba = lgbm.predict_proba(X_val)[:, 1]
    oof_proba_raw[val_idx] = raw_proba

    pr_auc = average_precision_score(y_val, raw_proba)
    auroc  = roc_auc_score(y_val, raw_proba)
    n_pos  = int(y_val.sum())
    fold_results.append({
        'fold': fold + 1,
        'pr_auc': pr_auc,
        'auroc': auroc,
        'n_malignant_val': n_pos,
        'best_iteration': lgbm.best_iteration_,
        'n_train_after_smote': len(y_res),
    })
    print(f'Fold {fold+1}: PR-AUC={pr_auc:.4f}  AUROC={auroc:.4f}  '
          f'val_malignant={n_pos}  best_iter={lgbm.best_iteration_}  '
          f'train_size={len(y_res):,}')

fold_df = pd.DataFrame(fold_results)
print(f'\nMean PR-AUC: {fold_df.pr_auc.mean():.4f} \u00b1 {fold_df.pr_auc.std():.4f}')
print(f'Mean AUROC:  {fold_df.auroc.mean():.4f} \u00b1 {fold_df.auroc.std():.4f}')
print(f'Mean best_iteration: {np.mean(best_iterations):.0f}')

Fold 1: PR-AUC=0.0657  AUROC=0.9471  val_malignant=77  best_iter=266  train_size=342,739
Fold 2: PR-AUC=0.0298  AUROC=0.9372  val_malignant=83  best_iter=128  train_size=342,740
Fold 3: PR-AUC=0.0746  AUROC=0.9418  val_malignant=78  best_iter=23  train_size=342,740
Fold 4: PR-AUC=0.0329  AUROC=0.9455  val_malignant=78  best_iter=120  train_size=342,740
Fold 5: PR-AUC=0.0376  AUROC=0.9301  val_malignant=77  best_iter=331  train_size=342,740

Mean PR-AUC: 0.0481 ± 0.0205
Mean AUROC:  0.9403 ± 0.0069
Mean best_iteration: 174


In [9]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

folds = fold_df['fold'].values
for ax, metric, title in [
    (axes[0], 'pr_auc', 'PR-AUC per Fold'),
    (axes[1], 'auroc',  'AUROC per Fold'),
]:
    ax.bar(folds, fold_df[metric].values, color='steelblue', edgecolor='black')
    ax.axhline(fold_df[metric].mean(), color='crimson', linestyle='--',
               label=f'Mean = {fold_df[metric].mean():.4f}')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Fold')
    ax.set_ylabel(metric.upper().replace('_', '-'))
    ax.legend()
    ax.set_ylim(0, max(fold_df[metric].max() * 1.2, 0.5))

plt.tight_layout()
plt.savefig(SAVE_DIR / 'fold_results.png', dpi=150, bbox_inches='tight')
plt.close()
print('Fold results plot saved.')
print(fold_df.to_string(index=False))

Fold results plot saved.
 fold   pr_auc    auroc  n_malignant_val  best_iteration  n_train_after_smote
    1 0.065678 0.947084               77             266               342739
    2 0.029833 0.937195               83             128               342740
    3 0.074563 0.941806               78              23               342740
    4 0.032887 0.945512               78             120               342740
    5 0.037640 0.930065               77             331               342740


## Step 6 — Platt Scaling Calibration (Tuned C)

OOF predictions are truly out-of-sample, so fitting a logistic regression on top of them is a
valid calibration step. The regularisation strength `C` controls how aggressively the sigmoid
is fitted — too large (C → ∞) overfits the OOF raw scores; too small (C → 0) under-fits and
leaves probabilities biased.

We select `C` by **5-fold cross-validation on the OOF predictions**, minimising Brier score.
This is valid because the OOF set is fixed and the calibrator only sees raw probabilities as
input (a single feature), so 5-fold CV on it is low-variance and not subject to leakage.

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss
from sklearn.model_selection import cross_val_score

C_candidates = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
brier_by_C   = {}

print(f'{"C":>10} | {"Mean Brier (5-fold CV)":>22}')
print('-' * 37)
for C in C_candidates:
    lr     = LogisticRegression(C=C, max_iter=1000)
    scores = cross_val_score(
        lr,
        oof_proba_raw.reshape(-1, 1),
        y_model,
        cv=5,
        scoring='neg_brier_score',
    )
    brier_by_C[C] = -scores.mean()
    print(f'{C:>10} | {brier_by_C[C]:>22.8f}')

best_C = min(brier_by_C, key=brier_by_C.get)
print(f'\nSelected C = {best_C}  (lowest 5-fold Brier score)')

         C | Mean Brier (5-fold CV)
-------------------------------------
     0.001 |             0.00097893
      0.01 |             0.00097882
       0.1 |             0.00097407
       1.0 |             0.00098457
      10.0 |             0.00098746
     100.0 |             0.00098887

Selected C = 0.1  (lowest 5-fold Brier score)


In [11]:
calibrator    = LogisticRegression(C=best_C, max_iter=1000)
calibrator.fit(oof_proba_raw.reshape(-1, 1), y_model)
oof_proba_cal = calibrator.predict_proba(oof_proba_raw.reshape(-1, 1))[:, 1]

brier_raw  = brier_score_loss(y_model, oof_proba_raw)
brier_cal  = brier_score_loss(y_model, oof_proba_cal)
brier_base = brier_score_loss(y_model, np.full(len(y_model), y_model.mean()))

pr_auc_cal = average_precision_score(y_model, oof_proba_cal)
auroc_cal  = roc_auc_score(y_model, oof_proba_cal)

print('=== CALIBRATION RESULTS ===')
print(f'Brier score (raw):          {brier_raw:.6f}')
print(f'Brier score (calibrated):   {brier_cal:.6f}  (lower = better)')
print(f'Brier baseline (base rate): {brier_base:.6f}')
print(f'\nPR-AUC (calibrated OOF):    {pr_auc_cal:.4f}')
print(f'AUROC  (calibrated OOF):    {auroc_cal:.4f}')
print(f'\nTrue malignancy rate:         {y_model.mean():.6f}')
print(f'Mean calibrated probability:  {oof_proba_cal.mean():.6f}')
print(f'Raw prob range:  {oof_proba_raw.min():.4f} \u2013 {oof_proba_raw.max():.4f}')
print(f'Cal prob range:  {oof_proba_cal.min():.4f} \u2013 {oof_proba_cal.max():.4f}')

=== CALIBRATION RESULTS ===
Brier score (raw):          0.003178
Brier score (calibrated):   0.000972  (lower = better)
Brier baseline (base rate): 0.000979

PR-AUC (calibrated OOF):    0.0336
AUROC  (calibrated OOF):    0.8774

True malignancy rate:         0.000980
Mean calibrated probability:  0.000959
Raw prob range:  0.0002 – 0.8733
Cal prob range:  0.0008 – 0.1144


In [12]:
from sklearn.calibration import calibration_curve

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, proba, label, color in [
    (axes[0], oof_proba_raw, 'Raw (uncalibrated)', 'steelblue'),
    (axes[1], oof_proba_cal, f'Calibrated (Platt, C={best_C})', 'crimson'),
]:
    frac_pos, mean_pred = calibration_curve(y_model, proba, n_bins=20, strategy='quantile')
    ax.plot(mean_pred, frac_pos, 's-', color=color, label=label)
    ax.plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
    ax.set_xlabel('Mean predicted probability')
    ax.set_ylabel('Fraction actually malignant')
    ax.set_title(f'Calibration Curve \u2014 {label}', fontweight='bold')
    ax.legend()

plt.tight_layout()
plt.savefig(SAVE_DIR / 'calibration_curve.png', dpi=150, bbox_inches='tight')
plt.close()
print('Calibration curve saved.')

Calibration curve saved.


## Step 7 — Risk Tier Threshold Selection

### Two thresholds, three tiers

| Boundary | Method | What it optimises |
|---|---|---|
| **Low \| Medium** | Youden's J on ROC curve | Maximises `TPR − FPR` — statistically optimal for a single binary cut |
| **Medium \| High** | Minimum-recall constraint on PR curve | Clinical goal: High tier must capture at least X% of all malignant cases |

### Why Youden's J for the Low|Medium boundary?

Youden's J (`TPR − FPR`) is the threshold that maximises the sum of sensitivity and specificity.
It is the mathematically optimal operating point for a symmetric loss function. For the Low|Medium
boundary — where we are deciding "is this worth elevated concern?" — symmetry is reasonable:
missing a malignant case and over-flagging a benign case carry roughly equal downstream cost at
this tier.

### Why a recall constraint for the Medium|High boundary?

The High tier has a distinct clinical purpose: it should flag cases where **immediate specialist
referral is warranted**. This demands the highest achievable PPV — we want most records in this
tier to actually be malignant. But PPV and recall trade off: a stricter threshold raises PPV and
shrinks recall. The recall constraint makes this tradeoff explicit.

### Sensitivity analysis

The table below shows the consequences of five different minimum-recall targets for the High tier.
Read each row as: "If I require the High tier to catch at least X% of all malignant cases,
what threshold, tier size, and PPV does that give me?"

In [13]:
from sklearn.metrics import roc_curve, precision_recall_curve

fpr, tpr, roc_thresh = roc_curve(y_model, oof_proba_cal)
prec, rec, pr_thresh = precision_recall_curve(y_model, oof_proba_cal)

# Youden's J — Low|Medium boundary (fixed, independent of recall target choice)
j_scores       = tpr - fpr
best_j_idx     = np.argmax(j_scores)
youdens_thresh = float(roc_thresh[best_j_idx])
sensitivity_j  = tpr[best_j_idx]
specificity_j  = 1 - fpr[best_j_idx]

total_malignant = int(y_model.sum())

print(f"Low|Medium boundary (Youden's J): {youdens_thresh:.6f}")
print(f'  Sensitivity: {sensitivity_j:.1%}  Specificity: {specificity_j:.1%}\n')

# Sensitivity analysis for Medium|High boundary
recall_targets = [0.30, 0.40, 0.50, 0.60, 0.75]
analysis_rows  = []

header = (f'{"Recall target":>14} | {"High threshold":>15} | '
          f'{"High tier size":>14} | {"Sensitivity":>12} | {"PPV":>10}')
print(header)
print('-' * len(header))

for target in recall_targets:
    # rec[:-1] aligns with pr_thresh (precision_recall_curve convention)
    valid = np.where(rec[:-1] >= target)[0]
    if len(valid) == 0:
        print(f'{target:>13.0%}  | (no valid threshold found at this recall level)')
        continue

    high_thresh = float(pr_thresh[valid[-1]])
    # Enforce High boundary is strictly above Medium boundary
    if high_thresh <= youdens_thresh:
        high_thresh = youdens_thresh * 1.001

    high_mask  = oof_proba_cal >= high_thresh
    n_high     = int(high_mask.sum())
    n_high_mal = int(y_model[high_mask].sum())
    sens       = n_high_mal / total_malignant
    ppv        = n_high_mal / n_high if n_high > 0 else 0.0

    analysis_rows.append({
        'recall_target':    target,
        'high_threshold':   round(high_thresh, 6),
        'high_tier_size':   n_high,
        'high_sensitivity': round(sens, 4),
        'high_ppv':         round(ppv, 6),
    })
    print(f'{target:>13.0%}  | {high_thresh:>15.6f} | {n_high:>14,} | {sens:>11.1%} | {ppv:>9.3%}')

df_thresh_analysis = pd.DataFrame(analysis_rows)
df_thresh_analysis.to_csv(RESULTS_DIR / 'v3_threshold_sensitivity.csv', index=False)
print('\nSensitivity analysis saved.')

Low|Medium boundary (Youden's J): 0.001058
  Sensitivity: 74.0%  Specificity: 90.5%

 Recall target |  High threshold | High tier size |  Sensitivity |        PPV
-----------------------------------------------------------------------------
          30%  |        0.003736 |          2,625 |       30.0% |    4.495%
          40%  |        0.002692 |          4,182 |       40.2% |    3.778%
          50%  |        0.001850 |          7,481 |       50.1% |    2.633%
          60%  |        0.001379 |         13,560 |       60.1% |    1.740%
          75%  |        0.001059 |         37,924 |       73.8% |    0.765%

Sensitivity analysis saved.


### Threshold decision

**We choose a minimum recall target of 50% for the High tier.**

Reasoning:

1. **Clinical floor**: A High-risk tier that catches fewer than half of all malignant cases would
   fail its core purpose — the majority of truly dangerous lesions would be downgraded to Medium
   or Low and potentially missed. 50% is the minimum clinically defensible sensitivity for a
   tier meant to trigger immediate referral.

2. **PPV vs. system load tradeoff**: Moving from 50% → 60% recall roughly doubles the High tier
   size (see table above) while reducing PPV, meaning twice as many benign cases are subjected
   to unnecessary specialist workup. The 50% point offers the best sensitivity-to-workload ratio.

3. **Asymmetric cost**: False negatives (missed malignant) are clinically worse than false
   positives (unnecessary referral), which argues for higher recall. But the Youden's J
   Medium tier already catches the next tranche of malignant cases — patients in Medium are
   not ignored, they receive standard follow-up. Only cases below Youden's J threshold are
   classified as Low risk.

**To change this decision**: update `CHOSEN_RECALL_TARGET` in the cell below. The thresholds,
tier assignments, and all downstream outputs will update automatically.

In [14]:
# === CLINICAL DECISION — change this value to re-run with a different recall target ===
CHOSEN_RECALL_TARGET = 0.50

chosen_row = df_thresh_analysis[df_thresh_analysis['recall_target'] == CHOSEN_RECALL_TARGET]
if chosen_row.empty:
    raise ValueError(f'No analysis row found for recall target {CHOSEN_RECALL_TARGET}. '
                     'Check that this value is in recall_targets above.')

threshold_high   = float(chosen_row['high_threshold'].iloc[0])   # Medium|High boundary
threshold_medium = youdens_thresh                                  # Low|Medium boundary

print('=== APPLIED THRESHOLDS ===')
print(f'High  risk  >= {threshold_high:.6f}  '
      f'(recall target: {CHOSEN_RECALL_TARGET:.0%}, '
      f'actual sensitivity: {chosen_row["high_sensitivity"].iloc[0]:.1%}, '
      f'PPV: {chosen_row["high_ppv"].iloc[0]:.3%})')
print(f'Medium risk   {threshold_medium:.6f} – {threshold_high:.6f}  '
      f"(Youden's J: sensitivity={sensitivity_j:.1%}, specificity={specificity_j:.1%})")
print(f'Low   risk  < {threshold_medium:.6f}')

=== APPLIED THRESHOLDS ===
High  risk  >= 0.001850  (recall target: 50%, actual sensitivity: 50.1%, PPV: 2.633%)
Medium risk   0.001058 – 0.001850  (Youden's J: sensitivity=74.0%, specificity=90.5%)
Low   risk  < 0.001058


In [15]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ROC curve
axes[0].plot(fpr, tpr, color='steelblue', lw=2, label=f'AUROC = {auroc_cal:.4f}')
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random')
axes[0].axvline(
    x=fpr[best_j_idx], color='orange', linestyle=':',
    label=f"Youden's J (Low|Medium boundary, TPR={sensitivity_j:.2f})"
)
axes[0].set_xlabel('False Positive Rate (1 - Specificity)')
axes[0].set_ylabel('True Positive Rate (Sensitivity)')
axes[0].set_title('ROC Curve', fontweight='bold')
axes[0].legend(fontsize=9)

# PR curve
axes[1].plot(rec, prec, color='steelblue', lw=2, label=f'PR-AUC = {pr_auc_cal:.4f}')
axes[1].axhline(
    y=y_model.mean(), color='gray', linestyle='--',
    label=f'Baseline (prevalence = {y_model.mean():.4%})'
)
axes[1].axvline(
    x=CHOSEN_RECALL_TARGET, color='crimson', linestyle=':',
    label=f'{CHOSEN_RECALL_TARGET:.0%} recall (Medium|High boundary)'
)
axes[1].set_xlabel('Recall (Sensitivity)')
axes[1].set_ylabel('Precision (PPV)')
axes[1].set_title('Precision-Recall Curve', fontweight='bold')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig(SAVE_DIR / 'roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.close()
print('ROC and PR curves saved.')

ROC and PR curves saved.


## Step 8 — Assign Risk Tiers

Calibrated OOF probabilities are out-of-sample (never seen during training or calibration fitting),
so the tier assignments are unbiased estimates of real-world performance.

In [16]:
def assign_tier(p):
    if p >= threshold_high:
        return 'High'
    elif p >= threshold_medium:
        return 'Medium'
    return 'Low'

df['malignancy_proba'] = oof_proba_cal
df['risk_tier']        = df['malignancy_proba'].map(assign_tier)

TIER_ORDER  = ['High', 'Medium', 'Low']
TIER_COLORS = ['crimson', 'orange', 'steelblue']

print('=== RISK TIER SUMMARY ===')
print(f'{"Tier":<8} {"Records":>10} {"Malignant":>10} {"Sensitivity":>13} {"PPV":>10}')
print('-' * 57)
tier_rows = []
for tier in TIER_ORDER:
    mask   = df['risk_tier'] == tier
    n_tot  = int(mask.sum())
    n_mal  = int(df.loc[mask, 'malignant'].sum())
    sens   = n_mal / total_malignant
    ppv    = n_mal / n_tot if n_tot > 0 else 0.0
    print(f'{tier:<8} {n_tot:>10,} {n_mal:>10} {sens:>12.1%} {ppv:>10.4%}')
    tier_rows.append({'tier': tier, 'n_records': n_tot, 'n_malignant': n_mal,
                      'sensitivity': round(sens, 4), 'ppv': round(ppv, 6)})

print(f'\nTotal malignant captured: {sum(r["n_malignant"] for r in tier_rows)}')

=== RISK TIER SUMMARY ===
Tier        Records  Malignant   Sensitivity        PPV
---------------------------------------------------------
High          7,477        196        49.9%    2.6214%
Medium       30,795         95        24.2%    0.3085%
Low         362,787        102        26.0%    0.0281%

Total malignant captured: 393


In [17]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

tier_n = [df[df['risk_tier'] == t].shape[0] for t in TIER_ORDER]
axes[0].bar(TIER_ORDER, tier_n, color=TIER_COLORS)
axes[0].set_title('Records per Risk Tier', fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(tier_n):
    axes[0].text(i, v + 2000, f'{v:,}', ha='center', fontsize=9, fontweight='bold')

mal_n = [int(df[df['risk_tier'] == t]['malignant'].sum()) for t in TIER_ORDER]
axes[1].bar(TIER_ORDER, mal_n, color=TIER_COLORS)
axes[1].set_title('Malignant Cases per Risk Tier', fontweight='bold')
axes[1].set_ylabel('Count')
for i, v in enumerate(mal_n):
    axes[1].text(i, v + 2, f'{v}\n({v/total_malignant:.0%})', ha='center',
                 fontsize=9, fontweight='bold')

for tier, color in zip(TIER_ORDER, TIER_COLORS):
    subset = df[df['risk_tier'] == tier]['malignancy_proba']
    axes[2].hist(subset, bins=50, alpha=0.6, label=tier, color=color, density=True)
axes[2].set_xlabel('Calibrated Probability')
axes[2].set_ylabel('Density')
axes[2].set_title('Probability Distribution by Tier', fontweight='bold')
axes[2].legend()
axes[2].axvline(x=threshold_high,   color='crimson', linestyle='--', lw=1.5)
axes[2].axvline(x=threshold_medium, color='orange',  linestyle='--', lw=1.5)

plt.tight_layout()
plt.savefig(SAVE_DIR / 'risk_tier_distribution.png', dpi=150, bbox_inches='tight')
plt.close()
print('Risk tier distribution saved.')

Risk tier distribution saved.


## Step 9 — SHAP Explainability

SHAP is disabled by default (runs ~10–30 min on 400k rows). To enable it:
1. Uncomment the SHAP cells below.
2. The final LightGBM model from Step 10 is the correct model to pass to `TreeExplainer` —
   it is trained on all data, so its feature attributions represent the full population.

In [18]:
# import shap
# explainer   = shap.TreeExplainer(final_lgbm)  # run after Step 10
# shap_values = explainer.shap_values(X_model)
# sv = shap_values[1] if isinstance(shap_values, list) else shap_values
print('SHAP skipped.')

SHAP skipped.


In [19]:
# shap.summary_plot(sv, X_model, show=False, max_display=15)
# plt.tight_layout()
# plt.savefig(SAVE_DIR / 'shap_beeswarm.png', dpi=150, bbox_inches='tight')
# plt.close()
# shap.summary_plot(sv, X_model, plot_type='bar', show=False, max_display=15)
# plt.tight_layout()
# plt.savefig(SAVE_DIR / 'shap_importance.png', dpi=150, bbox_inches='tight')
# plt.close()
print('SHAP plots skipped.')

SHAP plots skipped.


## Step 10 — Train Final Model & Save

### Why train a separate final model?

The CV models above are each trained on 4/5 of the data. For inference on new patients we want
a model trained on **all available data**. We train it with the Optuna best parameters and
`n_estimators` set to the mean of the CV folds' early-stopping iterations (with a 10% buffer)
— this avoids over-fitting since we no longer have a validation set for early stopping.

### Calibrator transfer

The Platt calibrator was fitted on OOF raw scores. The final model's raw scores will have a
similar distribution (same architecture, same training data, just no held-out fold) so the
calibrator transfers without re-fitting. At deployment time, pass new samples through
`final_lgbm → calibrator → thresholds` to get risk tiers.

### Saved artefacts

| File | Contents |
|---|---|
| `v3_lgbm.pkl` | Final LightGBM model (trained on full data) |
| `v3_calibrator.pkl` | Platt scaling logistic regression |
| `v3_thresholds.pkl` | `{'high': threshold_high, 'medium': threshold_medium}` |
| `v3_feature_cols.pkl` | Ordered list of feature column names |
| `v3_best_params.pkl` | Optuna best hyperparameters (for reproducibility) |

In [22]:
import joblib

# n_estimators = mean best_iteration across CV folds + 10% buffer
final_n_estimators = int(np.mean(best_iterations) * 1.10)
print(f'CV mean best_iteration: {np.mean(best_iterations):.0f}')
print(f'Final n_estimators (x1.10 buffer): {final_n_estimators}')

smote_final  = SMOTE(
    sampling_strategy=best_params['sampling_strategy'],
    random_state=42,
    k_neighbors=5,
)
X_final_res, y_final_res = smote_final.fit_resample(X_model, y_model)
print(f'Full training set after SMOTE: {len(y_final_res):,} rows '
      f'(malignant: {y_final_res.sum():,})')

final_lgbm = lgb.LGBMClassifier(
    n_estimators=final_n_estimators,
    learning_rate=best_params['learning_rate'],
    num_leaves=best_params['num_leaves'],
    min_child_samples=best_params['min_child_samples'],
    scale_pos_weight=best_params['scale_pos_weight'],
    subsample=best_params['subsample'],
    colsample_bytree=best_params['colsample_bytree'],
    reg_alpha=best_params['reg_alpha'],
    reg_lambda=best_params['reg_lambda'],
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)
final_lgbm.fit(X_final_res, y_final_res)
print('Final model trained.')

CV mean best_iteration: 174
Final n_estimators (x1.10 buffer): 190
Full training set after SMOTE: 428,425 rows (malignant: 27,759)
Final model trained.


In [23]:
feature_cols = list(X_model.columns)

joblib.dump(final_lgbm,  MODEL_DIR / 'v3_lgbm.pkl')
joblib.dump(calibrator,  MODEL_DIR / 'v3_calibrator.pkl')
joblib.dump({'high': threshold_high, 'medium': threshold_medium},
            MODEL_DIR / 'v3_thresholds.pkl')
joblib.dump(feature_cols, MODEL_DIR / 'v3_feature_cols.pkl')
joblib.dump(best_params,  MODEL_DIR / 'v3_best_params.pkl')

print('=== MODEL ARTEFACTS SAVED ===')
for f in sorted(MODEL_DIR.glob('*.pkl')):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:<30} {size_kb:>8.1f} KB')

# Full scored dataset
df.to_csv(DATA_PROCESSED / 'dataset_v3_scored.csv', index=False)

# Risk tier summary
pd.DataFrame(tier_rows).to_csv(RESULTS_DIR / 'v3_risk_tiers.csv', index=False)

# Full model metrics
metrics = {
    'mean_pr_auc_raw':      fold_df['pr_auc'].mean(),
    'std_pr_auc_raw':       fold_df['pr_auc'].std(),
    'mean_auroc_raw':       fold_df['auroc'].mean(),
    'std_auroc_raw':        fold_df['auroc'].std(),
    'pr_auc_calibrated':    pr_auc_cal,
    'auroc_calibrated':     auroc_cal,
    'brier_raw':            brier_raw,
    'brier_calibrated':     brier_cal,
    'brier_baseline':       brier_base,
    'platt_C':              best_C,
    'threshold_high':       threshold_high,
    'threshold_medium':     threshold_medium,
    'chosen_recall_target': CHOSEN_RECALL_TARGET,
    'sensitivity_at_high':  float(chosen_row['high_sensitivity'].iloc[0]),
    'specificity_at_medium': specificity_j,
    'final_n_estimators':   final_n_estimators,
    'optuna_best_pr_auc':   best_value,
    **{f'param_{k}': v for k, v in best_params.items()},
}
pd.DataFrame([metrics]).round(6).to_csv(RESULTS_DIR / 'v3_model_metrics.csv', index=False)

print('\n=== ALL OUTPUTS SAVED ===')
print(f'  {DATA_PROCESSED / "dataset_v3_scored.csv"}')
print(f'  {RESULTS_DIR / "v3_risk_tiers.csv"}')
print(f'  {RESULTS_DIR / "v3_model_metrics.csv"}')
print(f'  {RESULTS_DIR / "v3_threshold_sensitivity.csv"}')
print(f'  Graphs: {SAVE_DIR}/')
print(f'  Models: {MODEL_DIR}/')

=== MODEL ARTEFACTS SAVED ===
  v3_best_params.pkl                  0.2 KB
  v3_calibrator.pkl                   0.9 KB
  v3_feature_cols.pkl                 1.7 KB
  v3_lgbm.pkl                      1964.5 KB
  v3_thresholds.pkl                   0.0 KB

=== ALL OUTPUTS SAVED ===
  /home/xcane/dev/ml/170melanoma/data/processed/v3/dataset_v3_scored.csv
  /home/xcane/dev/ml/170melanoma/outputs/results/v3/v3_risk_tiers.csv
  /home/xcane/dev/ml/170melanoma/outputs/results/v3/v3_model_metrics.csv
  /home/xcane/dev/ml/170melanoma/outputs/results/v3/v3_threshold_sensitivity.csv
  Graphs: /home/xcane/dev/ml/170melanoma/outputs/graphs/v3/
  Models: /home/xcane/dev/ml/170melanoma/outputs/models/v3/


## Inference Usage

To score new patient records after this notebook has been run once:

```python
import joblib, pandas as pd

model        = joblib.load('outputs/models/v3/v3_lgbm.pkl')
calibrator   = joblib.load('outputs/models/v3/v3_calibrator.pkl')
thresholds   = joblib.load('outputs/models/v3/v3_thresholds.pkl')
feature_cols = joblib.load('outputs/models/v3/v3_feature_cols.pkl')

# df_new: new patient records, same columns as training data (after preprocessing)
raw_proba = model.predict_proba(df_new[feature_cols])[:, 1]
cal_proba = calibrator.predict_proba(raw_proba.reshape(-1, 1))[:, 1]

def assign_risk(p):
    if p >= thresholds['high']:   return 'High'
    if p >= thresholds['medium']: return 'Medium'
    return 'Low'

df_new['malignancy_proba'] = cal_proba
df_new['risk_tier']        = [assign_risk(p) for p in cal_proba]
```